In [1]:
"""
Statistical Significance for Brain Signal Classification
=========================================================
Python implementation of the methods described in:

Combrisson & Jerbi (2015) - "Exceeding chance level by chance: The caveat of
theoretical chance levels in brain signal classification and statistical
assessment of decoding accuracy"
Journal of Neuroscience Methods, 250, 126-136.

Two approaches are implemented:
  1. Binomial cumulative distribution (analytical)
  2. Permutation tests (empirical/data-driven)

Plus simulation utilities to reproduce the paper's core findings.
"""

import numpy as np
from scipy.stats import binom
from scipy.stats import norm as sp_norm
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, LeaveOneOut, cross_val_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")


# =============================================================================
# 1. BINOMIAL-BASED SIGNIFICANCE THRESHOLD
# =============================================================================
def age(ind):
    if '3m' in ind:
        return 'young'
    else:
        return 'old'

def condition(ind):
    if '5xFAD' in ind:
        return '5xFAD'
    else:
        return 'Wild-Type'


def binomial_threshold(n_samples: int, n_classes: int, alpha: float = 0.05) -> float:
    """
    Compute the minimum decoding accuracy (%) required to be statistically
    significant, using the binomial cumulative distribution.

    This accounts for finite sample size — unlike the naive theoretical
    chance level of 100/n_classes %.

    Parameters
    ----------
    n_samples : int
        Total number of test samples.
    n_classes : int
        Number of classes (e.g. 2 for binary classification).
    alpha : float
        Significance level (e.g. 0.05, 0.01, 0.001).

    Returns
    -------
    float
        Minimum accuracy (%) that is statistically significant at level alpha.

    Example
    -------
    >>> binomial_threshold(n_samples=40, n_classes=2, alpha=0.001)
    75.0
    # Matches the paper's Table 1: for n=40, 2-class, p<0.001 → 75%
    """
    chance = 1.0 / n_classes
    # binom.ppf gives the value k such that P(X <= k) >= 1 - alpha
    k = binom.ppf(1 - alpha, n_samples, chance)
    threshold = (k / n_samples) * 100.0
    return round(threshold, 1)


def significance_table(
    sample_sizes: list = None,
    n_classes_list: list = None,
    alphas: list = None,
) -> dict:
    """
    Reproduce Table 1 from the paper: a lookup table of significance thresholds.

    Parameters
    ----------
    sample_sizes : list of int
        Sample sizes to evaluate. Defaults to paper values.
    n_classes_list : list of int
        Numbers of classes. Defaults to [2, 4, 8].
    alphas : list of float
        Significance levels. Defaults to [0.05, 0.01, 0.001, 0.0001].

    Returns
    -------
    dict
        Nested dict: result[n][c][alpha] = threshold (%).
    """
    table = {}
    for n in sample_sizes:
        table[n] = {}
        for c in n_classes_list:
            table[n][c] = {}
            for a in alphas:
                table[n][c][a] = binomial_threshold(n, c, a)
    return table


def print_significance_table(table: dict = None) -> None:
    """Pretty-print the significance threshold table (replicates Table 1)."""
    if table is None:
        table = significance_table()

    sample_sizes = sorted(table.keys())
    alphas = [0.05, 0.01, 0.001, 0.0001]
    alpha_labels = ["p<0.05", "p<0.01", "p<0.001", "p<0.0001"]

    for c in [2, 4, 8]:
        print(f"\n{'='*60}")
        print(f"  {c}-CLASS CLASSIFICATION — Significance Thresholds (%)")
        print(f"{'='*60}")
        header = f"{'n':>6}  " + "  ".join(f"{l:>9}" for l in alpha_labels)
        print(header)
        print("-" * len(header))
        for n in sample_sizes:
            row = f"{n:>6}  " + "  ".join(
                f"{table[n][c][a]:>8.1f}%" for a in alphas
            )
            print(row)


# =============================================================================
# 2. PERMUTATION TEST
# =============================================================================

def permutation_test(
    X: np.ndarray,
    y: np.ndarray,
    classifier=None,
    n_permutations: int = 1000,
    cv_folds: int = 10,
    alpha: float = 0.05,
    random_state: int = 42,
) -> dict:
    """
    Assess classification significance via label permutation testing.

    Shuffles class labels repeatedly and computes the null distribution of
    classification accuracies. The original accuracy is then compared to
    this distribution to derive an empirical p-value.

    Parameters
    ----------
    X : np.ndarray, shape (n_samples, n_features)
        Feature matrix.
    y : np.ndarray, shape (n_samples,)
        Class labels.
    classifier : sklearn estimator, optional
        Defaults to LinearDiscriminantAnalysis.
    n_permutations : int
        Number of label permutations (paper used 10,000; 1,000 is faster).
    cv_folds : int
        Number of cross-validation folds.
    alpha : float
        Significance level for the threshold.
    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    dict with keys:
        - 'observed_accuracy': float — accuracy on original labels (%)
        - 'null_distribution': np.ndarray — accuracies under null (%)
        - 'p_value': float — empirical p-value
        - 'threshold': float — significance boundary at given alpha (%)
        - 'is_significant': bool

    """
    rng = np.random.default_rng(random_state)

    if classifier is None:
        classifier = LinearDiscriminantAnalysis()

    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)

    # Observed accuracy
    observed = np.mean(cross_val_score(classifier, X, y, cv=cv, scoring="accuracy"))

    # Null distribution via label permutations
    null_accuracies = np.zeros(n_permutations)
    for i in range(n_permutations):
        y_perm = rng.permutation(y)
        null_accuracies[i] = np.mean(
            cross_val_score(classifier, X, y_perm, cv=cv, scoring="accuracy")
        )

    # Empirical p-value: fraction of permutations >= observed accuracy
    p_value = np.mean(null_accuracies >= observed)

    # Significance threshold at given alpha
    threshold = np.percentile(null_accuracies, (1 - alpha) * 100)

    return {
        "observed_accuracy": observed * 100,
        "null_distribution": null_accuracies * 100,
        "p_value": p_value,
        "threshold": threshold * 100,
        "is_significant": observed > threshold,
    }



In [17]:
df['type']

MR-0644          WT_3m
MR-0645          WT_3m
MR-0648          WT_3m
MR-0649          WT_3m
MR-0654-t1       WT_3m
MR-0654-t2       WT_3m
MR-0655          WT_6m
MR-0656          WT_6m
MR-0657-t2       WT_6m
MR-0658          WT_6m
MR-0659-t1       WT_6m
MR-0659-t2    5xFAD_3m
MR-0661       5xFAD_3m
MR-0662       5xFAD_3m
MR-0663       5xFAD_3m
MR-0665       5xFAD_3m
MR-0667       5xFAD_3m
MR-0668       5xFAD_3m
MR-0669       5xFAD_6m
MR-0674       5xFAD_6m
MR-0676       5xFAD_6m
MR-0677       5xFAD_6m
MR-0678       5xFAD_6m
MR-0679       5xFAD_6m
Name: type, dtype: object

In [19]:
import json
import joblib
import numpy as np
import pandas as pd
# ── config ────────────────────────────────────────────────────────────────────

dataset_map = {
    'energy':   'Energy_level3.json',
    'coactiv':  'Coactiv.json',
    'entropy':  'Entropy.json',
    'kl_div':   'kl_div.json',
}

label_map = {
    'age':  age,        # your existing function
    'cond': condition,  # your existing function
}

# experiments you actually have models for
experiments = [5, 6, 9, 12, 14, 16]  # adjust to your actual model indices

models_dir = '1D_ind_sin_analyses/models/'

# ── load datasets once ────────────────────────────────────────────────────────

dataframes = {
    name: pd.read_json('1D_ind_sin_analyses/'+path)
    for name, path in dataset_map.items()
}

# ── permutation test loop ─────────────────────────────────────────────────────

results = {}

for dataset_name, df in dataframes.items():
    for label_name, label_fn in label_map.items():
        for exp in experiments:

            model_path = f'{models_dir}{dataset_name}_{label_name}_svm_{exp}.pkl'

            # skip if model doesn't exist for this combination
            try:
                svm = joblib.load(model_path)
            except FileNotFoundError:
                continue

            # features: column exp as 2D array (n_samples, 1)
            X = df[str(exp)].values.reshape(-1, 1)

            # labels: binary array from type column
            if label_name =="age":
                y = df['type'].apply(age)
            else:
                y = df['type'].apply(condition)

            result = permutation_test(
                X              = X,
                y              = y,
                classifier     = svm,
                n_permutations = 1000,
                cv_folds       = 10,
                alpha          = 0.05,
            )

            key = f'{dataset_name}_{label_name}_{exp}'
            results[key] = {
                'dataset':    dataset_name,
                'label':      label_name,
                'experiment': exp,
                'p_value':    result['p_value'],
                'observed':   result['observed_accuracy'],
                'threshold':  result['threshold'],
                'significant': result['is_significant'],
            }

            print(
                f"{key:<35} "
                f"acc={result['observed_accuracy']:.1f}%  "
                f"p={result['p_value']:.4f}  "
                f"sig={result['is_significant']}"
            )

# ── save results ──────────────────────────────────────────────────────────────

results_df = pd.DataFrame(results).T
results_df.to_csv('permutation_test_results.csv', index=True)
print("\nSaved: permutation_test_results.csv")

energy_age_5                        acc=71.7%  p=0.0210  sig=True
energy_age_6                        acc=56.7%  p=0.1240  sig=False
energy_age_9                        acc=78.3%  p=0.0050  sig=True
energy_age_12                       acc=71.7%  p=0.0540  sig=False
energy_age_14                       acc=76.7%  p=0.0160  sig=True
energy_age_16                       acc=81.7%  p=0.0060  sig=True
energy_cond_5                       acc=40.0%  p=0.7840  sig=False
energy_cond_6                       acc=78.3%  p=0.0210  sig=True
energy_cond_9                       acc=61.7%  p=0.2370  sig=False
energy_cond_12                      acc=38.3%  p=0.8280  sig=False
energy_cond_14                      acc=55.0%  p=0.3840  sig=False
energy_cond_16                      acc=43.3%  p=0.7790  sig=False


KeyError: '5'